# Table extraction comparison: *Attention Is All You Need*

Runs pdfplumber, PyMuPDF, tabula (lattice + stream), PaddleOCR PP-StructureV3 and Docling on the same PDF and puts their table outputs side by side.

pdfplumber, PyMuPDF and tabula reuse the logic from `backend/app/core/document_parser.py` (without the junk/reference filters, so raw outputs are visible).

In [ ]:
import os
import re
import time
from io import StringIO

import fitz
import pandas as pd

os.environ["PADDLE_PDX_DISABLE_MODEL_SOURCE_CHECK"] = "True"

PDF = "../data/attention_is_all_you_need.pdf"
results, timings = {}, {}


def run(name, extract):
    start = time.time()
    results[name] = extract()
    timings[name] = round(time.time() - start, 1)
    print(f"{name}: {len(results[name])} tables in {timings[name]}s")

## Ground truth
Pages whose text contains a `Table N:` caption.

In [ ]:
with fitz.open(PDF) as doc:
    PAGE_COUNT = doc.page_count
    ground_truth = {page.number + 1: re.findall(r"Table \d+:", page.get_text()) for page in doc}
{page: captions for page, captions in ground_truth.items() if captions}

## pdfplumber

In [ ]:
import pdfplumber


def extract_pdfplumber():
    tables = []
    with pdfplumber.open(PDF) as pdf:
        for page_num, page in enumerate(pdf.pages, 1):
            for table in page.extract_tables():
                if table:
                    df = pd.DataFrame(table[1:], columns=table[0])
                    tables.append((page_num, df.to_markdown(index=False)))
    return tables


run("pdfplumber", extract_pdfplumber)

## PyMuPDF

In [ ]:
def extract_pymupdf():
    tables = []
    with fitz.open(PDF) as doc:
        for page_num, page in enumerate(doc, 1):
            for table in page.find_tables():
                df = table.to_pandas()
                if not df.empty:
                    tables.append((page_num, df.to_markdown(index=False)))
    return tables


run("pymupdf", extract_pymupdf)

## tabula (needs Java)

In [ ]:
import tabula


def extract_tabula(mode):
    tables = []
    for page_num in range(1, PAGE_COUNT + 1):
        for df in tabula.read_pdf(PDF, pages=page_num, multiple_tables=True, silent=True, **{mode: True}):
            if not df.empty:
                tables.append((page_num, df.to_markdown(index=False)))
    return tables


run("tabula-lattice", lambda: extract_tabula("lattice"))
run("tabula-stream", lambda: extract_tabula("stream"))

## PaddleOCR PP-StructureV3 (light config for Mac CPU)
Mobile OCR models, medium layout model, SLANet_plus table structure, and end-to-end table recognition (skips the two RT-DETR-L cell detectors). Time includes model loading.

In [ ]:
from paddleocr import PPStructureV3


def extract_paddle():
    pipeline = PPStructureV3(
        layout_detection_model_name="PP-DocLayout-M",
        text_detection_model_name="PP-OCRv5_mobile_det",
        text_recognition_model_name="PP-OCRv5_mobile_rec",
        wired_table_structure_recognition_model_name="SLANet_plus",
        wireless_table_structure_recognition_model_name="SLANet_plus",
        use_doc_orientation_classify=False,
        use_doc_unwarping=False,
        use_textline_orientation=False,
        use_region_detection=False,
        use_formula_recognition=False,
        use_chart_recognition=False,
        use_seal_recognition=False,
        device="cpu",
    )
    tables = []
    output = pipeline.predict(
        input=PDF,
        use_e2e_wired_table_rec_model=True,
        use_e2e_wireless_table_rec_model=True,
    )
    for page_num, res in enumerate(output, 1):
        for table in res["table_res_list"]:
            df = pd.read_html(StringIO(table["pred_html"]))[0]
            tables.append((page_num, df.to_markdown(index=False)))
    return tables


run("paddle", extract_paddle)

## Docling
Default pipeline. Time includes model loading.

In [ ]:

from docling.document_converter import DocumentConverter


def extract_docling():
    doc = DocumentConverter().convert(PDF).document
    return [(t.prov[0].page_no, t.export_to_dataframe(doc=doc).to_markdown(index=False)) for t in doc.tables]


run("docling", extract_docling)

In [ ]:
import sys; print(sys.prefix)


## Summary
Tables found per page by each tool, and wall-clock seconds.

In [ ]:
found = pd.DataFrame(
    [(name, page) for name, tables in results.items() for page, _ in tables],
    columns=["tool", "page"],
)
display(pd.crosstab(found.page, found.tool, margins=True))
pd.Series(timings, name="seconds")

## Side-by-side outputs
Every tool's output for each page that has a real table.

In [ ]:
def show(page):
    print(f"########## Page {page}: {ground_truth[page]} ##########\n")
    for name, tables in results.items():
        outputs = [md for p, md in tables if p == page]
        print(f"===== {name}: {len(outputs)} table(s) =====")
        for md in outputs:
            print(md, end="\n\n")


for page, captions in ground_truth.items():
    if captions:
        show(page)